In [1]:
import pandas as pd

df = pd.read_excel('data_2_MCU.xlsx', engine='openpyxl')
missing_value = df.isnull().sum()
cols_to_drop = missing_value[missing_value > 3000].index
df.drop(columns=cols_to_drop, inplace=True)
df.head(10)
df.to_csv(
    'data_2_MCU.csv', index= False
)
df.set_index('TANGGAL', inplace=True)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.to_list()
cat_df = pd.DataFrame({'Column': categorical_cols, 'Unique Values': [df[col].nunique() for col in categorical_cols]}).set_index('Column')
cat_df.sort_values(by='Unique Values', ascending=False, inplace=True)
cat_binary_encode = [col for col in categorical_cols if cat_df.loc[col, 'Unique Values'] <= 2]
cat_df_to_drop = [col for col in categorical_cols if cat_df.loc[col, 'Unique Values'] > 100]
df.drop(columns=cat_df_to_drop, inplace=True)
df.sample(10)


,BADGE,KEHAMILAN,OLAHRAGA,ALERGI,TINGGI,BERAT,TENSI,NADI,PERNAPASAN,SUHU,...,TRIGLISERIDA,HDL_KOLEST,LDL_KOLEST,UREUM,KREATININ,ASAM_URAT_GINJAL,GULA_DARAH_PUASA,GULA_DARAH_2JAMPP,URINE_REDUKSI_PUASA,URINE_REDUKSI_2JAMPP
TANGGAL,,,,,,,,,,,,,,,,,,,,,
2023-03-06,132591590.0,-,+,-,1720.0,630.0,110/70,570.0,200.0,360.0,...,1300.0,440.0,1160.0,300.0,1100.0,7500.0,830.0,810.0,Negatif,Negatif
2020-12-03,132790150.0,-,+,-,1700.0,730.0,100/70,650.0,200.0,360.0,...,1840.0,400.0,1070.0,190.0,800.0,5500.0,880.0,840.0,Negatif,Negatif
2020-02-22,133187120.0,-,+,-,1670.0,670.0,100/70,740.0,200.0,360.0,...,1510.0,560.0,1160.0,170.0,900.0,5900.0,820.0,830.0,Negatif,Negatif
2024-03-04,133314320.0,-,-,-,1700.0,960.0,140/80,630.0,200.0,360.0,...,1260.0,540.0,1670.0,180.0,10.0,7300.0,900.0,850.0,Negatif,Negatif
2025-04-22,132590180.0,-,+,-,1630.0,630.0,120/80,540.0,200.0,360.0,...,910.0,680.0,1720.0,220.0,600.0,4100.0,890.0,890.0,Negatif,Negatif
2024-04-17,133305830.0,-,+,-,1690.0,550.0,120/80,580.0,200.0,360.0,...,1060.0,720.0,1880.0,200.0,800.0,4200.0,1120.0,860.0,Negatif,Negatif
2023-03-10,133306540.0,-,+,-,1640.0,640.0,120/70,620.0,200.0,360.0,...,1370.0,400.0,1280.0,170.0,800.0,7300.0,950.0,850.0,Negatif,Negatif
2022-08-15,133402050.0,-,+,-,1780.0,870.0,120/90,800.0,200.0,360.0,...,1300.0,360.0,1760.0,190.0,800.0,7900.0,910.0,1000.0,Negatif,Negatif
2022-08-05,132903470.0,NaN,+,-,1570.0,690.0,120/80,570.0,200.0,360.0,...,1020.0,660.0,1270.0,140.0,700.0,50.0,810.0,780.0,Negatif,Negatif


In [2]:
df.drop(columns= [col for col in cat_binary_encode if df[col].nunique()==1], inplace=True)
df.groupby(by='BADGE').size().sort_values(ascending=False).median()
numeric_cols = df.select_dtypes(include=['number']).columns.to_list()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
agg_df = df.groupby(by='BADGE').agg(
    {**{col: 'median' for col in numeric_cols},
     **{col: lambda x: x.mode()[0] if not x.mode().empty else None for col in categorical_cols}}
)
print(list(agg_df.columns))
agg_df = agg_df.astype({col: 'float64' for col in ['BADGE']}) 
agg_df = agg_df.loc[:, ~agg_df.columns.duplicated()]
agg_df.reset_index(drop=True, inplace=True)
agg_df.to_csv('Agg_pasien_MCU_2.csv', index=False)


['BADGE', 'TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'HB', 'LEUKOSIT', 'LED', 'EOSINOPIL', 'BASOPIL', 'SEGMENT', 'LYMPOSIT', 'MONOSIT', 'TROMBOSIT', 'BILIRUBIN_TOTAL', 'BILIRUBIN_DIRECT', 'BILIRUBIN_INDIRECT', 'ALKALINE_PHOSPAT', 'SGPT', 'SGOT', 'GAMMA_GT', 'KOLEST_TOTAL', 'TRIGLISERIDA', 'HDL_KOLEST', 'LDL_KOLEST', 'UREUM', 'KREATININ', 'ASAM_URAT_GINJAL', 'GULA_DARAH_PUASA', 'GULA_DARAH_2JAMPP', 'KEHAMILAN', 'OLAHRAGA', 'ALERGI', 'TENSI', 'KULIT_RAMBUT', 'PENYAKIT_MATA', 'CONJUNGTIVA', 'SCLERA', 'TELINGA', 'MEMBRAN_TYMPANI', 'REFLEK_CAHAYA', 'SERUMEN_PLUG', 'HIDUNG', 'SEPTUM_DEVIASI', 'CONCHA', 'POLYP', 'KERONGKONGAN', 'TONSIL', 'FARING', 'HERNIA', 'HAEMORROID', 'EPIDIDYMIS_TESTIS_PROSTAT', 'LEHER', 'JVP', 'STRUMA', 'MULUT', 'GUSI', 'BATAS_JANTUNG', 'IRAMA_JANTUNG', 'SUARA_JANTUNG_MURMUR', 'AUSKULTASI', 'THORAX_PHOTO', 'TREMOR', 'SEMBAB', 'PARALYSE', 'PATELA', 'DINDING_PERUT', 'PERUT', 'HATI', 'LIMPA', 'PROTEIN_ALBUMIN', 'REDUKSI', 'UROBILINOGEN', 'BILIRUBIN', 'ERITROSIT_RBC', 

In [3]:
agg_df.columns

Index(['BADGE', 'TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'HB',
       'LEUKOSIT', 'LED', 'EOSINOPIL', 'BASOPIL', 'SEGMENT', 'LYMPOSIT',
       'MONOSIT', 'TROMBOSIT', 'BILIRUBIN_TOTAL', 'BILIRUBIN_DIRECT',
       'BILIRUBIN_INDIRECT', 'ALKALINE_PHOSPAT', 'SGPT', 'SGOT', 'GAMMA_GT',
       'KOLEST_TOTAL', 'TRIGLISERIDA', 'HDL_KOLEST', 'LDL_KOLEST', 'UREUM',
       'KREATININ', 'ASAM_URAT_GINJAL', 'GULA_DARAH_PUASA',
       'GULA_DARAH_2JAMPP', 'KEHAMILAN', 'OLAHRAGA', 'ALERGI', 'TENSI',
       'KULIT_RAMBUT', 'PENYAKIT_MATA', 'CONJUNGTIVA', 'SCLERA', 'TELINGA',
       'MEMBRAN_TYMPANI', 'REFLEK_CAHAYA', 'SERUMEN_PLUG', 'HIDUNG',
       'SEPTUM_DEVIASI', 'CONCHA', 'POLYP', 'KERONGKONGAN', 'TONSIL', 'FARING',
       'HERNIA', 'HAEMORROID', 'EPIDIDYMIS_TESTIS_PROSTAT', 'LEHER', 'JVP',
       'STRUMA', 'MULUT', 'GUSI', 'BATAS_JANTUNG', 'IRAMA_JANTUNG',
       'SUARA_JANTUNG_MURMUR', 'AUSKULTASI', 'THORAX_PHOTO', 'TREMOR',
       'SEMBAB', 'PARALYSE', 'PATELA', 'DINDING_PERUT', 'PERU

In [4]:
import category_encoders as ce

cat_cols = agg_df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    agg_df[col] = agg_df[col].astype(str).str.strip()
    agg_df[col] = agg_df[col].replace({
        'tidak diperiksa': 'Tidak diperiksa',
        'tidak periksa': 'Tidak diperiksa',
        'tidak di periksa': 'Tidak diperiksa',
        'tidak diperiksa ': 'Tidak diperiksa'
    })

print("BINARY COLUMNS (nunique == 2)")
binary_cols = {}
for col in cat_cols:
    unique_vals = agg_df[col].dropna().unique()
    if len(unique_vals) == 2:
        binary_cols[col] = unique_vals.tolist()
        print(col, unique_vals.tolist())

print("\nMULTI-CARDINALITY COLUMNS (nunique > 2)")
multi_card_cols = {}
for col in cat_cols:
    unique_vals = agg_df[col].dropna().unique()
    if len(unique_vals) > 2:
        multi_card_cols[col] = unique_vals.tolist()
        print(col, unique_vals.tolist())

binary_mapping = {
    'OLAHRAGA': {'+': 1, '-': 0},
    'ALERGI': {'+': 1, '-': 0},
    'CONJUNGTIVA': {'normal': 0, 'anemis': 1},
    'HAEMORROID': {'normal': 0, 'ada': 1},
    'LEHER': {'normal': 0, 'pembesaran KGB': 1},
    'IRAMA_JANTUNG': {'normal': 0, 'Ireguler': 1},
    'TREMOR': {'normal': 0, 'ada': 1},
    'PERUT': {'normal': 0, 'nyeri ketok VA kiri': 1},
    'LIMPA': {'normal': 0, 'teraba': 1},
    'UROBILINOGEN': {'Negatif': 1, 'Tidak periksa': 0}
}

ordinal_mapping = {
    'ERITROSIT_RBC': {'Negatif': 0, '+': 1, '++': 2, '+++': 3},
    'LEKOSIT_WBC': {'Negatif': 0, '+': 1, '++': 2, '+++': 3},
    'SEL_EPITEL': {'Negatif': 0, '+': 1, '++': 2, '+++': 3},
}

for col, mapping in binary_mapping.items():
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].map(mapping)

for col, mapping in ordinal_mapping.items():
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].map(mapping)

remaining_cat_cols = agg_df.select_dtypes(include=['object', 'category']).columns.tolist()
if remaining_cat_cols:
    encoder = ce.BinaryEncoder(cols=remaining_cat_cols, return_df=True)
    agg_df = encoder.fit_transform(agg_df)


BINARY COLUMNS (nunique == 2)
OLAHRAGA ['+', '-']
ALERGI ['-', '+']
CONJUNGTIVA ['normal', 'anemis']
HAEMORROID ['normal', 'ada']
EPIDIDYMIS_TESTIS_PROSTAT ['normal', 'None']
LEHER ['normal', 'pembesaran KGB']
IRAMA_JANTUNG ['normal', 'Ireguler']
TREMOR ['normal', 'ada']
PERUT ['normal', 'nyeri ketok VA kiri']
LIMPA ['normal', 'teraba']
UROBILINOGEN ['Negatif', 'Tidak periksa']
BILIRUBIN ['Negatif', 'Tidak periksa']
ASAM_URAT_URIN ['Negatif', 'Tidak periksa']
TRIPLE_PHOSP ['Negatif', 'Tidak periksa']
HYALINE ['Negatif', 'Tidak periksa']

MULTI-CARDINALITY COLUMNS (nunique > 2)
KEHAMILAN ['-', 'G1P0A0 hamil 15 mgg', 'None', 'G1 P0 A0', 'G4 P2 A1', 'G2P1A0 hamil 31 mgg (+)', 'G2 P1 A0', 'G2P0A1 Hamil 7 bulan', 'G3P2A0 hamil 30 mggu', 'G5P3A1 Hamil 25-26 minggu JTH', 'G2P1A0', 'G3P2A0 hamil 8 mggu', 'G1P0A0 hamil 25 mgg', 'G2P1A0 hamil 29 mggu (rutin kontrol dr.mustofa SpOG)', 'G5P2A2 hamil 7 mgg', 'G3 P2 A0', 'G3P2A0 hamil 26 mgg', 'G1PoAo ( hamil 24 minggu )']
TENSI ['110/70', '120/80',

In [7]:
import pandas as pd
import numpy as np
import re
import category_encoders as ce

encoded_df = agg_df.copy()

# ===================================================================
# 1. TENSI → SISTOLIK & DIASTOLIK
# ===================================================================
def parse_tensi(x):
    if pd.isna(x): return np.nan, np.nan
    s = str(x).strip().replace(' ', '')
    if '/' not in s: return np.nan, np.nan
    a, b = s.split('/', 1)
    try:
        return float(a) if a else np.nan, float(b) if b else np.nan
    except:
        return np.nan, np.nan

if 'TENSI' in encoded_df.columns:
    encoded_df[['SISTOLIK', 'DIASTOLIK']] = encoded_df['TENSI'].apply(
        lambda x: pd.Series(parse_tensi(x))
    )
    encoded_df.drop('TENSI', axis=1, inplace=True)

# ===================================================================
# 2. KEHAMILAN → TRIMESTER (0 = tidak hamil, 1/2/3)
# ===================================================================
def parse_trimester(x):
    if pd.isna(x): return 0
    s = str(x).lower()
    if '-' in s or 'none' in s or not s: return 0
    m = re.search(r'(\d+)\s*(minggu|mgg|bulan)', s)
    if not m: return 0
    num = int(m.group(1))
    weeks = num * 4.3 if 'bulan' in m.group(2) else num
    return 1 if weeks <= 13 else 2 if weeks <= 28 else 3

if 'KEHAMILAN' in encoded_df.columns:
    encoded_df['TRIMESTER'] = encoded_df['KEHAMILAN'].apply(parse_trimester)
    encoded_df.drop('KEHAMILAN', axis=1, inplace=True)

# ===================================================================
# 3. URINE MICROSCOPIC (ERITROSIT_RBC, LEKOSIT_WBC, SEL_EPITEL)
# ===================================================================
def parse_microscopic(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    if s in ['', 'none', 'tidak periksa', 'tidak_diperiksa', '-', 'tidak diperiksa']:
        return np.nan

    # range atau angka tunggal
    nums = re.findall(r'\d+\.?\d*', s)
    if nums:
        nums = [float(n) for n in nums]
        return np.mean(nums)

    # kata-kata umum
    mapping = {
        'nihil':0, 'tidak ada':0, 'negatif':0,
        'sedikit':1, 'rar':1,
        '+':3, '++':8, '+++':15,
        'banyak':12, 'penuh':20, 'padat':20
    }
    return mapping.get(s, np.nan)

urin_cols = ['ERITROSIT_RBC', 'LEKOSIT_WBC', 'SEL_EPITEL']
for c in urin_cols:
    if c in encoded_df.columns:
        encoded_df[c] = encoded_df[c].apply(parse_microscopic)

# ===================================================================
# 4. SEMUA KOLOM OBJECT → lower + strip + standarisasi "tidak diperiksa"
# ===================================================================
obj_cols = encoded_df.select_dtypes(include=['object', 'category']).columns
for c in obj_cols:
    encoded_df[c] = (encoded_df[c]
                     .astype(str)
                     .str.strip()
                     .str.lower()
                     .replace({'tidak diperiksa':'tidak_diperiksa',
                               'tidak periksa':'tidak_diperiksa',
                               'vtidak diperiksa':'tidak_diperiksa',
                               'none':'tidak_diperiksa',
                               'nan':'tidak_diperiksa',
                               'tidak melakukan':'tidak_diperiksa'}))

# ===================================================================
# 5. BINARY & ORDINAL MAPPING (sesuai nilai unik yang kamu berikan)
# ===================================================================
binary_map = {
    'OLAHRAGA'                   : {'+':1, '-':0, 'tidak_diperiksa':np.nan},
    'ALERGI'                     : {'+':1, '-':0, 'tidak_diperiksa':np.nan},
    'CONJUNGTIVA'                : {'normal':0, 'anemis':1, 'tidak_diperiksa':np.nan},
    'HAEMORROID'                 : {'normal':0, 'ada':1, 'tidak_diperiksa':np.nan},
    'EPIDIDYMIS_TESTIS_PROSTAT'  : {'normal':0, 'tidak_diperiksa':np.nan},
    'LEHER'                      : {'normal':0, 'pembesaran kgb':1, 'tidak_diperiksa':np.nan},
    'IRAMA_JANTUNG'              : {'normal':0, 'ireguler':1, 'tidak_diperiksa':np.nan},
    'TREMOR'                     : {'normal':0, 'ada':1, 'tidak_diperiksa':np.nan},
    'PERUT'                      : {'normal':0, 'nyeri ketok va kiri':1, 'tidak_diperiksa':np.nan},
    'LIMPA'                      : {'normal':0, 'teraba':1, 'tidak_diperiksa':np.nan},
}

ordinal_map = {
    'UROBILINOGEN'    : {'negatif':0, 'tidak_diperiksa':np.nan},
    'BILIRUBIN'       : {'negatif':0, 'tidak_diperiksa':np.nan},
    'ASAM_URAT_URIN'  : {'negatif':0, 'tidak_diperiksa':np.nan},
    'TRIPLE_PHOSP'    : {'negatif':0, 'tidak_diperiksa':np.nan},
    'HYALINE'         : {'negatif':0, 'tidak_diperiksa':np.nan},
    'PROTEIN_ALBUMIN' : {'negatif':0, 'positif 1':1, 'positif':2, 'tidak_diperiksa':np.nan},
    'REDUKSI'         : {'negatif':0, 'positif 1':1, 'positif 2':2, 'positif':2, 'tidak_diperiksa':np.nan},
    'AMORF'           : {'negatif':0, 'positif':1, 'tidak_diperiksa':np.nan},
    'CA_OX'           : {'negatif':0, 'positif':1, 'tidak_diperiksa':np.nan},
    'GRANULER'        : {'negatif':0, 'positif':1, 'tidak_diperiksa':np.nan},
    'BAKTERI'         : {'negatif':0, 'positif':1, 'positif 2':2, 'tidak_diperiksa':np.nan},
    'URINE_REDUKSI_PUASA': {'negatif':0, 'positif1':1, 'positif 1':1, 'positif 2':2, 'positif':2, 'tab':np.nan, 'tidak_diperiksa':np.nan},
    'URINE_REDUKSI_2JAMPP': {'negatif':0, 'positif 1':1, 'positif 2':2, 'positif 3':3, 'positif':2, 'tab':np.nan, 'tidak_diperiksa':np.nan},
}

# terapkan mapping (aman untuk kolom yang sudah numerik)
for col, mp in {**binary_map, **ordinal_map}.items():
    if col in encoded_df.columns:
        # jika sudah 0/1, biarkan
        if encoded_df[col].dtype in ['int64','float64'] and encoded_df[col].dropna().isin([0,1]).all():
            continue
        encoded_df[col] = encoded_df[col].map(mp)

# ===================================================================
# 6. KOLOM LAIN YANG MASIH KATEGORIKAL → Binary Encoding
# ===================================================================
# hapus kolom yang seluruhnya NaN dulu
cat_cols = encoded_df.select_dtypes(include=['object','category']).columns
encoded_df.drop(columns=[c for c in cat_cols if encoded_df[c].isna().all()], inplace=True)

# binary encoding untuk sisanya
remaining_cat = encoded_df.select_dtypes(include=['object','category']).columns.tolist()
if remaining_cat:
    encoder = ce.BinaryEncoder(cols=remaining_cat, return_df=True)
    encoded_df = encoder.fit_transform(encoded_df)

# ===================================================================
# DONE
# ===================================================================
print("Encoding selesai tanpa warning!")
print(f"Shape akhir: {encoded_df.shape}")
print(f"Missing di urin cols:")
print(encoded_df[['ERITROSIT_RBC','LEKOSIT_WBC','SEL_EPITEL']].isna().sum())

Encoding selesai tanpa warning!
Shape akhir: (1887, 144)
Missing di urin cols:
ERITROSIT_RBC    1887
LEKOSIT_WBC      1887
SEL_EPITEL       1887
dtype: int64


In [ ]:
encoded_df

,BADGE,TINGGI,BERAT,NADI,PERNAPASAN,SUHU,HB,LEUKOSIT,LED,EOSINOPIL,...,BAKTERI_0,BAKTERI_1,BAKTERI_2,URINE_REDUKSI_PUASA_0,URINE_REDUKSI_PUASA_1,URINE_REDUKSI_PUASA_2,URINE_REDUKSI_2JAMPP_0,URINE_REDUKSI_2JAMPP_1,URINE_REDUKSI_2JAMPP_2,URINE_REDUKSI_2JAMPP_3
345,132683840.0,1690.0,790.0,640.0,200.0,360.0,13500.0,7800.0,30.0,0.0,...,0,0,1,0,0,1,0,0,0,1
1753,166907880.0,1750.0,820.0,650.0,200.0,360.0,15400.0,4800.0,160.0,20.0,...,0,0,1,0,0,1,0,0,0,1
492,132699570.0,1680.0,680.0,790.0,200.0,360.0,15300.0,6500.0,120.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1305,133490720.0,1570.0,480.0,680.0,200.0,360.0,12800.0,7800.0,310.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1689,166277460.0,1600.0,650.0,750.0,200.0,360.0,130.0,5600.0,460.0,10.0,...,0,0,1,0,0,1,0,0,0,1
228,132598120.0,1600.0,750.0,840.0,200.0,360.0,13200.0,6300.0,360.0,20.0,...,0,0,1,0,0,1,0,0,0,1
760,132801440.0,1650.0,700.0,510.0,200.0,360.0,12500.0,5600.0,200.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1167,133192560.0,1610.0,875.0,755.0,200.0,360.0,11800.0,6350.0,300.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1868,167522660.0,1540.0,630.0,800.0,200.0,360.0,14100.0,6500.0,190.0,10.0,...,0,0,1,0,1,0,0,0,1,1
335,132682690.0,1670.0,750.0,720.0,200.0,360.0,14700.0,7100.0,250.0,20.0,...,0,0,1,0,0,1,0,0,0,1


In [9]:
encoded_df = encoded_df.drop(columns=['ERITROSIT_RBC', 'LEKOSIT_WBC', 'SEL_EPITEL'])
encoded_df.sample(10)

,BADGE,TINGGI,BERAT,NADI,PERNAPASAN,SUHU,HB,LEUKOSIT,LED,EOSINOPIL,...,BAKTERI_0,BAKTERI_1,BAKTERI_2,URINE_REDUKSI_PUASA_0,URINE_REDUKSI_PUASA_1,URINE_REDUKSI_PUASA_2,URINE_REDUKSI_2JAMPP_0,URINE_REDUKSI_2JAMPP_1,URINE_REDUKSI_2JAMPP_2,URINE_REDUKSI_2JAMPP_3
535,132773950.0,1675.0,790.0,775.0,200.0,360.0,14250.0,6050.0,185.0,10.0,...,0,0,1,0,0,1,0,0,0,1
1593,134211840.0,1730.0,850.0,750.0,200.0,360.0,140.0,6800.0,100.0,10.0,...,0,0,1,0,0,1,0,0,0,1
1351,133586130.0,1630.0,690.0,690.0,200.0,360.0,12800.0,4500.0,90.0,10.0,...,0,0,1,0,0,1,0,0,0,1
400,132689620.0,1590.0,660.0,775.0,200.0,360.0,14950.0,8050.0,125.0,15.0,...,0,0,1,0,0,1,0,0,0,1
231,132598780.0,1650.0,620.0,750.0,200.0,360.0,13800.0,5950.0,110.0,10.0,...,0,0,1,0,0,1,0,0,0,1
531,132773460.0,1730.0,700.0,690.0,200.0,360.0,16400.0,5900.0,30.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1450,133597370.0,1700.0,740.0,660.0,200.0,18280.0,15550.0,7650.0,105.0,15.0,...,0,0,1,0,0,1,0,0,0,1
724,132797990.0,1625.0,605.0,785.0,200.0,360.0,12100.0,5400.0,285.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1077,133004510.0,1690.0,810.0,735.0,200.0,360.0,14550.0,6450.0,90.0,20.0,...,0,0,1,0,0,1,0,0,0,1
737,132799060.0,1720.0,780.0,650.0,200.0,360.0,15600.0,4700.0,130.0,20.0,...,0,0,1,0,0,1,0,0,0,1


In [10]:
encoded_df = encoded_df.drop(columns=['KEHAMILAN_0', 'KEHAMILAN_1', 'KEHAMILAN_2', 'KEHAMILAN_3', 'KEHAMILAN_4'])
encoded_df.sample(10)

,BADGE,TINGGI,BERAT,NADI,PERNAPASAN,SUHU,HB,LEUKOSIT,LED,EOSINOPIL,...,BAKTERI_0,BAKTERI_1,BAKTERI_2,URINE_REDUKSI_PUASA_0,URINE_REDUKSI_PUASA_1,URINE_REDUKSI_PUASA_2,URINE_REDUKSI_2JAMPP_0,URINE_REDUKSI_2JAMPP_1,URINE_REDUKSI_2JAMPP_2,URINE_REDUKSI_2JAMPP_3
612,132784530.0,1680.0,930.0,960.0,200.0,360.0,17100.0,10700.0,50.0,20.0,...,0,0,1,0,0,1,0,0,0,1
55,132497190.0,1830.0,930.0,615.0,200.0,360.0,15650.0,5850.0,190.0,10.0,...,0,0,1,0,0,1,0,0,0,1
578,132780970.0,1650.0,650.0,720.0,200.0,360.0,15300.0,60.0,150.0,20.0,...,0,0,1,0,0,1,0,0,0,1
391,132688730.0,1650.0,630.0,640.0,200.0,360.0,14300.0,5800.0,120.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1218,133306720.0,1640.0,680.0,600.0,200.0,360.0,16100.0,8700.0,120.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1404,133592560.0,1660.0,650.0,655.0,200.0,360.0,14650.0,5050.0,50.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1225,133307250.0,1665.0,675.0,715.0,200.0,360.0,15050.0,7200.0,20.0,20.0,...,0,0,1,0,0,1,0,0,0,1
1363,133587600.0,1550.0,590.0,600.0,200.0,360.0,13650.0,9250.0,90.0,10.0,...,0,0,1,0,0,1,0,0,0,1
838,132809720.0,1650.0,790.0,780.0,200.0,360.0,15400.0,6900.0,160.0,10.0,...,0,0,1,0,0,1,0,0,0,1
1660,134223650.0,1675.0,675.0,760.0,200.0,360.0,12800.0,8550.0,180.0,25.0,...,0,0,1,0,0,1,0,0,0,1


In [20]:
sample_data = encoded_df.loc[:0]

# 1. Mengubah row menjadi list nilai
sample_list = sample_data.values.tolist()[0]  # ambil [0] karena cuma 1 row
print(sample_list)

[131862880.0, 1720.0, 730.0, 550.0, 200.0, 360.0, 16650.0, 2885.0, 40.0, 15.0, 0.0, 710.0, 225.0, 55.0, 1910.0, 1045.0, 305.0, 740.0, 720.0, 240.0, 175.0, 225.0, 1685.0, 1495.0, 425.0, 1120.0, 200.0, 455.0, 6700.0, 900.0, 810.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]


In [23]:
print(encoded_df.columns.tolist())

['BADGE', 'TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'HB', 'LEUKOSIT', 'LED', 'EOSINOPIL', 'BASOPIL', 'SEGMENT', 'LYMPOSIT', 'MONOSIT', 'TROMBOSIT', 'BILIRUBIN_TOTAL', 'BILIRUBIN_DIRECT', 'BILIRUBIN_INDIRECT', 'ALKALINE_PHOSPAT', 'SGPT', 'SGOT', 'GAMMA_GT', 'KOLEST_TOTAL', 'TRIGLISERIDA', 'HDL_KOLEST', 'LDL_KOLEST', 'UREUM', 'KREATININ', 'ASAM_URAT_GINJAL', 'GULA_DARAH_PUASA', 'GULA_DARAH_2JAMPP', 'OLAHRAGA', 'ALERGI', 'TENSI_0', 'TENSI_1', 'TENSI_2', 'TENSI_3', 'TENSI_4', 'KULIT_RAMBUT_0', 'PENYAKIT_MATA_0', 'CONJUNGTIVA', 'SCLERA_0', 'TELINGA_0', 'TELINGA_1', 'MEMBRAN_TYMPANI_0', 'MEMBRAN_TYMPANI_1', 'MEMBRAN_TYMPANI_2', 'REFLEK_CAHAYA_0', 'REFLEK_CAHAYA_1', 'REFLEK_CAHAYA_2', 'SERUMEN_PLUG_0', 'SERUMEN_PLUG_1', 'SERUMEN_PLUG_2', 'SERUMEN_PLUG_3', 'SERUMEN_PLUG_4', 'HIDUNG_0', 'HIDUNG_1', 'SEPTUM_DEVIASI_0', 'SEPTUM_DEVIASI_1', 'SEPTUM_DEVIASI_2', 'CONCHA_0', 'CONCHA_1', 'CONCHA_2', 'POLYP_0', 'POLYP_1', 'POLYP_2', 'KERONGKONGAN_0', 'KERONGKONGAN_1', 'TONSIL_0', 'TONSIL_1', 'TO

In [24]:
# CELL BARU — AGGREGATE BINARY ENCODED COLUMNS SUPAYA LEBIH RINGKAS & BAGUS BUAT CLUSTERING
final_df = encoded_df.copy()

# ===================================================================
# 1. BUANG KOLOM YANG HAMPIR SEMUA 0 (rare event) → tidak informatif
# ===================================================================
rare_threshold = 0.01  # <1% yang 1 → buang
rare_cols = final_df.columns[final_df.mean() < rare_threshold]
print(f"Buang {len(rare_cols)} kolom rare (<1% positif): {list(rare_cols)[:10]}...")
final_df = final_df.drop(columns=rare_cols)

# ===================================================================
# 2. GABUNG BINARY ENCODED KOLOM YANG SEJENIS (MANUAL GROUPING LOGIS)
# ===================================================================
# Kelompokkan berdasarkan organ / pemeriksaan
groupings = {
    # TELINGA & TURUNANNYA
    'ANY_EAR_ABNORMAL'       : ['TELINGA_1', 'MEMBRAN_TYMPANI_1', 'MEMBRAN_TYMPANI_2', 'SERUMEN_PLUG_1', 'SERUMEN_PLUG_2', 'SERUMEN_PLUG_3', 'SERUMEN_PLUG_4'],
    # HIDUNG & TURUNANNYA
    'ANY_NOSE_ABNORMAL'      : ['HIDUNG_1', 'SEPTUM_DEVIASI_1', 'SEPTUM_DEVIASI_2', 'CONCHA_1', 'CONCHA_2', 'POLYP_1', 'POLYP_2'],
    # MULUT & GIGI
    'ANY_ORAL_ABNORMAL'      : ['MULUT_1', 'MULUT_2', 'GUSI_1'],
    # TENGGOROKAN
    'ANY_THROAT_ABNORMAL'    : ['KERONGKONGAN_1', 'TONSIL_1', 'TONSIL_2', 'TONSIL_3', 'TONSIL_4', 'FARING_1', 'FARING_2'],
    # THORAX PHOTO
    'ANY_THORAX_PHOTO_ABNORMAL': ['THORAX_PHOTO_1', 'THORAX_PHOTO_2', 'THORAX_PHOTO_3'],
    # URINE SEDIMEN (non-negatif)
    'ANY_URINE_SEDIMENT_POS' : ['PROTEIN_ALBUMIN_1', 'PROTEIN_ALBUMIN_2', 'REDUKSI_1', 'REDUKSI_2', 
                                'AMORF_1', 'CA_OX_1', 'CA_OX_2', 'GRANULER_1', 'HYALINE_1', 
                                'BAKTERI_1', 'BAKTERI_2'],
    # GULA URINE (puasa / 2jam pp)
    'ANY_GLUCOSE_URINE_POS'  : ['URINE_REDUKSI_PUASA_1', 'URINE_REDUKSI_PUASA_2', 
                                'URINE_REDUKSI_2JAMPP_1', 'URINE_REDUKSI_2JAMPP_2', 'URINE_REDUKSI_2JAMPP_3'],
}

for new_col, old_cols in groupings.items():
    cols_exist = [c for c in old_cols if c in final_df.columns]
    if cols_exist:
        final_df[new_col] = final_df[cols_exist].max(axis=1)  # 1 jika ada salah satu abnormal
        final_df = final_df.drop(columns=cols_exist)

# ===================================================================
# 3. KEEP YANG PENTING & SUDAH BAGUS (binary langsung)
# ===================================================================
keep_direct = ['OLAHR conlusion', 'ALERGI', 'CONJUNGTIVA', 'HAEMORROID', 'LEHER', 'IRAMA_JANTUNG', 
               'TREMOR', 'PERUT', 'LIMPA', 'HERNIA_0', 'EPIDIDYMIS_TESTIS_PROSTAT_0', 
               'UROBILINOGEN', 'BILIRUBIN_0', 'ASAM_URAT_URIN_0', 'TRIPLE_PHOSP_0']

keep_direct = [c for c in keep_direct if c in final_df.columns]

# ===================================================================
# 4. FINAL SELECTION: numeric + lab + yang di-keep + yang baru digabung
# ===================================================================
numeric_lab_cols = ['TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'HB', 'LEUKOSIT', 
                    'LED', 'EOSINOPIL', 'BASOPIL', 'SEGMENT', 'LYMPOSIT', 'MONOSIT', 'TROMBOSIT',
                    'BILIRUBIN_TOTAL', 'BILIRUBIN_DIRECT', 'BILIRUBIN_INDIRECT', 'ALKALINE_PHOSPAT',
                    'SGPT', 'SGOT', 'GAMMA_GT', 'KOLEST_TOTAL', 'TRIGLISERIDA', 'HDL_KOLEST', 
                    'LDL_KOLEST', 'UREUM', 'KREATININ', 'ASAM_URAT_GINJAL', 
                    'GULA_WAKTU', 'GULA_DARAH_2JAMPP', 
                    'SISTOLIK', 'DIASTOLIK', 'ERITROSIT_RBC', 'LEKOSIT_WBC', 'SEL_EPITEL']

numeric_lab_cols = [c for c in numeric_lab_cols if c in final_df.columns]

# Gabungkan semua
selected_cols = numeric_lab_cols + keep_direct + list(groupings.keys())
final_clustering_df = final_df[selected_cols].copy()

# Optional: tambah BMI
if 'TINGGI' in final_clustering_df.columns and 'BERAT' in final_clustering_df.columns:
    final_clustering_df['BMI'] = final_clustering_df['BERAT'] / ((final_clustering_df['TINGGI']/100)**2)

# Optional: tambah MAP (Mean Arterial Pressure)
if 'SISTOLIK' in final_clustering_df.columns and 'DIASTOLIK' in final_clustering_df.columns:
    final_clustering_df['MAP'] = final_clustering_df['DIASTOLIK'] + (final_clustering_df['SISTOLIK'] - final_clustering_df['DIASTOLIK']) / 3

print("SELESAI! Kolom siap clustering:")
print(f"→ Dari {encoded_df.shape[1]} kolom → {final_clustering_df.shape[1]} kolom")
print(f"→ Kolom akhir: {list(final_clustering_df.columns)}")

Buang 36 kolom rare (<1% positif): ['CONJUNGTIVA', 'MEMBRAN_TYMPANI_0', 'REFLEK_CAHAYA_0', 'SERUMEN_PLUG_0', 'SEPTUM_DEVIASI_0', 'CONCHA_0', 'POLYP_0', 'TONSIL_0', 'TONSIL_1', 'FARING_0']...
SELESAI! Kolom siap clustering:
→ Dari 136 kolom → 42 kolom
→ Kolom akhir: ['TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'HB', 'LEUKOSIT', 'LED', 'EOSINOPIL', 'BASOPIL', 'SEGMENT', 'LYMPOSIT', 'MONOSIT', 'TROMBOSIT', 'BILIRUBIN_TOTAL', 'BILIRUBIN_DIRECT', 'BILIRUBIN_INDIRECT', 'ALKALINE_PHOSPAT', 'SGPT', 'SGOT', 'GAMMA_GT', 'KOLEST_TOTAL', 'TRIGLISERIDA', 'HDL_KOLEST', 'LDL_KOLEST', 'UREUM', 'KREATININ', 'ASAM_URAT_GINJAL', 'GULA_DARAH_2JAMPP', 'ALERGI', 'HAEMORROID', 'HERNIA_0', 'EPIDIDYMIS_TESTIS_PROSTAT_0', 'UROBILINOGEN', 'ANY_EAR_ABNORMAL', 'ANY_NOSE_ABNORMAL', 'ANY_ORAL_ABNORMAL', 'ANY_THROAT_ABNORMAL', 'ANY_THORAX_PHOTO_ABNORMAL', 'ANY_URINE_SEDIMENT_POS', 'ANY_GLUCOSE_URINE_POS', 'BMI']


In [ ]:
final_df

,BADGE,TINGGI,BERAT,NADI,PERNAPASAN,SUHU,HB,LEUKOSIT,LED,EOSINOPIL,...,BILIRUBIN_1,ASAM_URAT_URIN_1,TRIPLE_PHOSP_1,ANY_EAR_ABNORMAL,ANY_NOSE_ABNORMAL,ANY_ORAL_ABNORMAL,ANY_THROAT_ABNORMAL,ANY_THORAX_PHOTO_ABNORMAL,ANY_URINE_SEDIMENT_POS,ANY_GLUCOSE_URINE_POS
982,132908430.0,1645.0,615.0,655.0,200.0,360.0,14000.0,5950.0,25.0,10.0,...,1,1,1,1,1,1,1,1,1,1
1550,134097940.0,1690.0,750.0,720.0,200.0,360.0,140.0,9100.0,50.0,10.0,...,1,1,1,1,1,1,1,1,1,1
1320,133492390.0,1720.0,900.0,680.0,200.0,360.0,15600.0,5700.0,40.0,20.0,...,1,1,1,1,1,1,1,1,1,1
184,132593610.0,1710.0,780.0,570.0,200.0,360.0,15700.0,6100.0,110.0,20.0,...,1,1,1,1,1,1,1,1,1,1
733,132798720.0,1740.0,530.0,700.0,200.0,360.0,16600.0,7700.0,30.0,40.0,...,1,1,1,1,1,1,1,1,1,1
1626,134219110.0,1680.0,660.0,640.0,200.0,360.0,14200.0,7200.0,160.0,10.0,...,1,1,1,1,1,1,1,1,1,1
182,132593430.0,1560.0,430.0,800.0,200.0,360.0,13300.0,5950.0,150.0,10.0,...,1,1,1,1,1,1,1,1,1,1
312,132680130.0,1700.0,620.0,720.0,200.0,360.0,13800.0,5900.0,100.0,20.0,...,1,1,1,1,1,1,1,1,1,1
1556,134098600.0,1655.0,655.0,770.0,200.0,360.0,15450.0,5850.0,175.0,20.0,...,1,1,1,1,1,1,1,1,1,1
97,132501850.0,1680.0,685.0,775.0,200.0,360.0,14700.0,4750.0,90.0,15.0,...,1,1,1,1,1,1,1,1,1,1


In [ ]:
one_type_only = [loc for loc in final_df.columns if final_df[loc].nunique()==1]
final_df = final_df.drop(columns=one_type_only)
final_df.sample(10)

,BADGE,TINGGI,BERAT,NADI,PERNAPASAN,SUHU,HB,LEUKOSIT,LED,EOSINOPIL,...,EPIDIDYMIS_TESTIS_PROSTAT_0,EPIDIDYMIS_TESTIS_PROSTAT_1,GUSI_0,UROBILINOGEN,BILIRUBIN_1,ASAM_URAT_URIN_1,TRIPLE_PHOSP_1,ANY_ORAL_ABNORMAL,ANY_THORAX_PHOTO_ABNORMAL,ANY_GLUCOSE_URINE_POS
1716,166285050.0,1650.0,660.0,730.0,200.0,360.0,14900.0,5100.0,60.0,20.0,...,0,1,1,1,1,1,1,1,1,1
1293,133489480.0,1720.0,770.0,700.0,200.0,360.0,14600.0,6900.0,110.0,10.0,...,0,1,0,1,1,1,1,1,1,1
1608,134217130.0,1740.0,950.0,820.0,200.0,360.0,15300.0,6200.0,60.0,10.0,...,0,1,1,1,1,1,1,1,1,1
785,132803940.0,1770.0,840.0,760.0,200.0,360.0,15400.0,6800.0,230.0,10.0,...,0,1,1,1,1,1,1,1,1,1
1656,134223160.0,1670.0,520.0,630.0,200.0,360.0,7120.0,7600.0,15.0,15.0,...,0,1,1,1,1,1,1,1,1,1
1840,167412980.0,1620.0,650.0,700.0,200.0,360.0,13900.0,8600.0,280.0,10.0,...,0,1,1,1,1,1,1,1,1,1
606,132783860.0,1610.0,750.0,740.0,200.0,360.0,15800.0,11500.0,20.0,20.0,...,0,1,1,1,1,1,1,1,1,1
723,132797910.0,1700.0,870.0,690.0,200.0,360.0,13400.0,6400.0,190.0,20.0,...,0,1,1,1,1,1,1,1,1,1
21,132084980.0,1560.0,570.0,805.0,200.0,360.0,11100.0,6550.0,220.0,15.0,...,1,0,1,1,1,1,1,1,1,1
860,132812200.0,1650.0,740.0,680.0,200.0,360.0,14900.0,5700.0,40.0,10.0,...,0,1,1,1,1,1,1,1,1,1


In [28]:
null_columns = [col for col in final_df.columns if final_df[col].isnull().sum() > 0]

for col in null_columns:
    if final_df[col].dtype in ['int64', 'float64']:  # numerik
        median_value = final_df[col].median()
        final_df[col].fillna(median_value, inplace=True)
    else:  # categorical / binary
        mode_value = final_df[col].mode()[0]
        final_df[col].fillna(mode_value, inplace=True)
final_df

C:\Users\Loq Gaming\AppData\Local\Temp\ipykernel_18868\1616573314.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df[col].fillna(median_value, inplace=True)


,BADGE,TINGGI,BERAT,NADI,PERNAPASAN,SUHU,HB,LEUKOSIT,LED,EOSINOPIL,...,EPIDIDYMIS_TESTIS_PROSTAT_0,EPIDIDYMIS_TESTIS_PROSTAT_1,GUSI_0,UROBILINOGEN,BILIRUBIN_1,ASAM_URAT_URIN_1,TRIPLE_PHOSP_1,ANY_ORAL_ABNORMAL,ANY_THORAX_PHOTO_ABNORMAL,ANY_GLUCOSE_URINE_POS
0,131862880.0,1720.0,730.0,550.0,200.0,360.0,16650.0,2885.0,40.0,15.0,...,0,1,0,1,1,1,1,1,1,1
1,131862910.0,1500.0,525.0,750.0,200.0,360.0,11600.0,5750.0,250.0,20.0,...,0,1,1,1,1,1,1,1,1,1
2,131862920.0,1780.0,770.0,560.0,200.0,360.0,13400.0,7400.0,80.0,20.0,...,0,1,1,1,1,1,1,1,1,1
3,131862930.0,1530.0,670.0,890.0,200.0,360.0,11600.0,9300.0,700.0,10.0,...,1,0,0,1,1,1,1,1,1,1
4,131862940.0,1520.0,565.0,710.0,200.0,360.0,11100.0,6800.0,250.0,20.0,...,0,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1882,167725030.0,1665.0,635.0,620.0,200.0,18280.0,7160.0,3755.0,230.0,55.0,...,0,1,1,1,1,1,1,1,1,1
1883,167725180.0,1800.0,920.0,530.0,200.0,360.0,15900.0,8100.0,300.0,30.0,...,0,1,1,1,1,1,1,1,1,1
1884,167725400.0,1670.0,830.0,800.0,200.0,360.0,15200.0,5300.0,60.0,0.0,...,0,1,1,1,1,1,1,1,1,1
1885,167725740.0,1665.0,750.0,540.0,200.0,360.0,13550.0,7050.0,145.0,5.0,...,0,1,1,1,1,1,1,1,1,1


In [29]:
final_df.isnull().sum()

BADGE                          0
TINGGI                         0
BERAT                          0
NADI                           0
PERNAPASAN                     0
SUHU                           0
HB                             0
LEUKOSIT                       0
LED                            0
EOSINOPIL                      0
BASOPIL                        0
SEGMENT                        0
LYMPOSIT                       0
MONOSIT                        0
TROMBOSIT                      0
BILIRUBIN_TOTAL                0
BILIRUBIN_DIRECT               0
BILIRUBIN_INDIRECT             0
ALKALINE_PHOSPAT               0
SGPT                           0
SGOT                           0
GAMMA_GT                       0
KOLEST_TOTAL                   0
TRIGLISERIDA                   0
HDL_KOLEST                     0
LDL_KOLEST                     0
UREUM                          0
KREATININ                      0
ASAM_URAT_GINJAL               0
GULA_DARAH_PUASA               0
GULA_DARAH

In [30]:
final_df.to_csv('Cleaned_agg_pasien_MCU_2.csv', index=False)

In [ ]:
final_df